# Physics-Informed Neural Networks for Soil Moisture Prediction
## Reproducibility Code — Nature Submission

This notebook reproduces all **figures** and **tables** reported in the paper:

**Requirements:** Run cells in order. Data files in `../data/` and weights in `../weights/`.

## 1. Setup: Imports and Dependencies

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
# Add current folder to sys.path so we can import our local module
import sys, os
sys.path.append(os.getcwd())
# Import the RK4 integrator from your module
from tfsolverMIconstrb_float64 import rk4_integration

# Paths relative to this notebook's location (FINAL_SUBMISSION/code/)
DATA_DIR = "../data"
WEIGHTS_DIR = "../weights"

2026-04-04 13:59:30.332166: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-04 13:59:30.347151: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775325570.369610   44875 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775325570.376058   44875 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775325570.390316   44875 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

## 2. Data Loading
Load field sensor data from the citrus production site in Fort Pierce, Florida.
- `campbell_1h.json` / `campbellv_1h.json`: Irrigation rates (h/h)
- `davis_1h.json` / `davisv_1h.json`: Rainfall (mm/h) and potential evapotranspiration ET₀ (mm/h)
- `irrimeter_1h.json` / `irrimeterv_1h.json`: Soil moisture (%VWC) at 5, 15, 25, 35, 45, 55 cm depth
Suffixed with `v` = validation (forecast) period.

In [2]:
# Load and prepare training data
campbell_train = pd.read_json(os.path.join(DATA_DIR, "campbell_1h.json"))
davis_train = pd.read_json(os.path.join(DATA_DIR, "davis_1h.json"))
irrimeter_train = pd.read_json(os.path.join(DATA_DIR, "irrimeter_1h.json"))

# Convert timestamps
for df in [campbell_train, davis_train, irrimeter_train]:
    df['timestamp'] = pd.to_datetime(df['timestamp'])

# Rename columns
campbell_train = campbell_train.rename(columns={"measurement": "irrigation"})
davis_train = davis_train.rename(columns={
    "measured_rain_mm_h": "rain",
    "potential_evapotranspiration_mm_h": "et0"
})

# Select irrimeter columns
irrimeter_train = irrimeter_train[["timestamp", "moisture_1", "moisture_2", "moisture_3",
                                    "moisture_4", "moisture_5", "moisture_6"]]

# Merge all data
df_train = pd.merge(davis_train[["timestamp", "rain", "et0"]],
                    campbell_train[["timestamp", "irrigation"]], on="timestamp", how="inner")
df_train = pd.merge(df_train, irrimeter_train, on="timestamp", how="inner")

# Process data
df_train["et0"] = df_train["et0"].clip(lower=0.0)
df_train["time_hours"] = (df_train["timestamp"] - df_train["timestamp"].min()).dt.total_seconds() / 3600.0

# Normalization constants
normalize = {"time_hours": 1, "rain": 30, "et0": 0.8, "irrigation": 1, "moisture": 40}

print(f"✓ Training data: {df_train.shape[0]} points, {(df_train['timestamp'].max() - df_train['timestamp'].min()).days} days")

✓ Training data: 840 points, 34 days


In [3]:
# Load and prepare validation data
campbell_val = pd.read_json(os.path.join(DATA_DIR, "campbellv_1h.json"))
davis_val = pd.read_json(os.path.join(DATA_DIR, "davisv_1h.json"))
irrimeter_val = pd.read_json(os.path.join(DATA_DIR, "irrimeterv_1h.json"))

# Convert timestamps
for df in [campbell_val, davis_val, irrimeter_val]:
    df['timestamp'] = pd.to_datetime(df['timestamp'])

# Rename columns
campbell_val = campbell_val.rename(columns={"measurement": "irrigation"})
davis_val = davis_val.rename(columns={
    "measured_rain_mm_h": "rain",
    "potential_evapotranspiration_mm_h": "et0"
})

# Select irrimeter columns
irrimeter_val = irrimeter_val[["timestamp", "moisture_1", "moisture_2", "moisture_3",
                                "moisture_4", "moisture_5", "moisture_6"]]

# Merge all data
df_val = pd.merge(davis_val[["timestamp", "rain", "et0"]],
                  campbell_val[["timestamp", "irrigation"]], on="timestamp", how="inner")
df_val = pd.merge(df_val, irrimeter_val, on="timestamp", how="inner")

# Process data
df_val["et0"] = df_val["et0"].clip(lower=0.0)
df_val["time_hours"] = (df_val["timestamp"] - df_val["timestamp"].min()).dt.total_seconds() / 3600.0

# Filter for specific time range
df_val = df_val[(df_val["time_hours"] >= 216) & (df_val["time_hours"] <= 288)]

# Calculate proportions for visualization
n_train, n_val = len(df_train), len(df_val)
width_train, width_val = n_train / (n_train + n_val), n_val / (n_train + n_val)

print(f"✓ Validation data: {n_val} points, {(df_val['timestamp'].max() - df_val['timestamp'].min()).days} days")
print(f"✓ Proportions: Training {width_train:.1%}, Validation {width_val:.1%} (ratio {n_train/n_val:.1f}:1)")

✓ Validation data: 73 points, 3 days
✓ Proportions: Training 92.0%, Validation 8.0% (ratio 11.5:1)


## 3. Data Preprocessing
Normalize inputs and prepare TensorFlow tensors for PINN evaluation.

In [4]:
# Prepare training data for PINN evaluation
# Apply normalization
for col in ["time_hours", "rain", "et0", "irrigation"]:
    df_train[f"{col}_scaled"] = df_train[col] / normalize.get(col.replace("_hours", ""), normalize[col])

for i in range(1, 7):
    df_train[f"moisture_{i}_scaled"] = df_train[f"moisture_{i}"] / normalize["moisture"]

df_train = df_train.sort_values("time_hours").dropna().reset_index(drop=True)

# Create tensors
t_train_tf = tf.constant(df_train["time_hours_scaled"].values.astype(np.float64), dtype=tf.float64)
inputs_train_tf = tf.constant(df_train[["irrigation_scaled", "rain_scaled", "et0_scaled"]].values.astype(np.float64), dtype=tf.float64)
states_train_tf = tf.constant(df_train[[f"moisture_{i}_scaled" for i in range(1, 7)]].values.astype(np.float64), dtype=tf.float64)

# Extract numpy arrays and initial conditions
time_train_np = t_train_tf.numpy()
inputs_train_np = inputs_train_tf.numpy()
states_train_np = states_train_tf.numpy()
m0_train_np = states_train_tf[0].numpy()

# Mid-step values for measured data
m_mid_train = states_train_tf[:-1]
in_mid_train = inputs_train_tf[:-1]
m_mid_train_np_denorm = m_mid_train.numpy() * normalize["moisture"]

print(f"✓ Training data prepared: {len(time_train_np)} time points, initial state shape {m0_train_np.shape}")

✓ Training data prepared: 840 time points, initial state shape (6,)


W0000 00:00:1775325572.525949   44875 gpu_device.cc:2341] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


## 4. PINN Model Architecture
The PINN spatially discretizes Richards' equation into 6 soil layers. Each layer is described by neural network branches representing:
- `f_i(m_i)`: vertical gravitational drainage
- `f_{i,i+1}(m_i, m_{i+1})`: inter-layer capillary/potential-driven flow
- `fr_i(r)`: direct rain infiltration per layer
- `S_i(m_i, ET₀)`: plant water uptake (sink)
- `fl_i(m_i)`: lateral outflow
- `f_irr(q_irr)`: irrigation inflow (surface layer only)

In [5]:
class TwoTankPINN(tf.keras.Model):
    def __init__(self, neurons=49):
        super(TwoTankPINN, self).__init__()
        glorot_init = tf.keras.initializers.GlorotNormal()

        def scaled_he_init(shape, dtype=None):
            return 0.5 * tf.keras.initializers.HeNormal()(shape, dtype=dtype)

        # layer 1 outflow branch
        self.f1_dense1 = tf.keras.layers.Dense(neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.f1_dense2 = tf.keras.layers.Dense(neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.f1_dense3 = tf.keras.layers.Dense(neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.f1_out = tf.keras.layers.Dense(1, activation='mish', kernel_initializer=scaled_he_init, use_bias=False)

        # layer 2 outflow branch
        self.f2_dense1 = tf.keras.layers.Dense(neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.f2_dense2 = tf.keras.layers.Dense(neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.f2_dense3 = tf.keras.layers.Dense(neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.f2_out = tf.keras.layers.Dense(1, activation='mish', kernel_initializer=scaled_he_init, use_bias=False)

        # layer 3 outflow branch
        self.f3_dense1 = tf.keras.layers.Dense(neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.f3_dense2 = tf.keras.layers.Dense(neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.f3_dense3 = tf.keras.layers.Dense(neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.f3_out = tf.keras.layers.Dense(1, activation='mish', kernel_initializer=scaled_he_init, use_bias=False)

        # layer 4 outflow branch
        self.f4_dense1 = tf.keras.layers.Dense(neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.f4_dense2 = tf.keras.layers.Dense(neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.f4_dense3 = tf.keras.layers.Dense(neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.f4_out = tf.keras.layers.Dense(1, activation='mish', kernel_initializer=scaled_he_init, use_bias=False)

        # layer 5 outflow branch
        self.f5_dense1 = tf.keras.layers.Dense(neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.f5_dense2 = tf.keras.layers.Dense(neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.f5_dense3 = tf.keras.layers.Dense(neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.f5_out = tf.keras.layers.Dense(1, activation='mish', kernel_initializer=scaled_he_init, use_bias=False)

        # layer 6 outflow branch
        self.f6_dense1 = tf.keras.layers.Dense(neurons, activation='tanh', kernel_initializer=glorot_init)
        self.f6_dense2 = tf.keras.layers.Dense(neurons, activation='tanh', kernel_initializer=glorot_init)
        self.f6_dense3 = tf.keras.layers.Dense(neurons, activation='tanh', kernel_initializer=glorot_init)
        self.f6_out = tf.keras.layers.Dense(1, activation='tanh', kernel_initializer=scaled_he_init)
        #################
        # Inter-tank flow branch
        self.f12_dense1 = tf.keras.layers.Dense(neurons, activation='tanh', kernel_initializer=glorot_init, use_bias=False)
        self.f12_dense2 = tf.keras.layers.Dense(neurons, activation='tanh', kernel_initializer=glorot_init, use_bias=False)
        self.f12_dense3 = tf.keras.layers.Dense(neurons, activation='tanh', kernel_initializer=glorot_init, use_bias=False)
        self.f12_out = tf.keras.layers.Dense(1, activation='tanh', kernel_initializer=glorot_init, use_bias=False)

        self.f23_dense1 = tf.keras.layers.Dense(neurons, activation='tanh', kernel_initializer=glorot_init, use_bias=False)
        self.f23_dense2 = tf.keras.layers.Dense(neurons, activation='tanh', kernel_initializer=glorot_init, use_bias=False)
        self.f23_dense3 = tf.keras.layers.Dense(neurons, activation='tanh', kernel_initializer=glorot_init, use_bias=False)
        self.f23_out = tf.keras.layers.Dense(1, activation='tanh', kernel_initializer=glorot_init, use_bias=False)

        self.f34_dense1 = tf.keras.layers.Dense(neurons, activation='tanh', kernel_initializer=glorot_init, use_bias=False)
        self.f34_dense2 = tf.keras.layers.Dense(neurons, activation='tanh', kernel_initializer=glorot_init, use_bias=False)
        self.f34_dense3 = tf.keras.layers.Dense(neurons, activation='tanh', kernel_initializer=glorot_init, use_bias=False)
        self.f34_out = tf.keras.layers.Dense(1, activation='tanh', kernel_initializer=glorot_init, use_bias=False)

        self.f45_dense1 = tf.keras.layers.Dense(neurons, activation='tanh', kernel_initializer=glorot_init, use_bias=False)
        self.f45_dense2 = tf.keras.layers.Dense(neurons, activation='tanh', kernel_initializer=glorot_init, use_bias=False)
        self.f45_dense3 = tf.keras.layers.Dense(neurons, activation='tanh', kernel_initializer=glorot_init, use_bias=False)
        self.f45_out = tf.keras.layers.Dense(1, activation='tanh', kernel_initializer=glorot_init, use_bias=False)

        self.f56_dense1 = tf.keras.layers.Dense(neurons, activation='tanh', kernel_initializer=glorot_init, use_bias=False)
        self.f56_dense2 = tf.keras.layers.Dense(neurons, activation='tanh', kernel_initializer=glorot_init, use_bias=False)
        self.f56_dense3 = tf.keras.layers.Dense(neurons, activation='tanh', kernel_initializer=glorot_init, use_bias=False)
        self.f56_out = tf.keras.layers.Dense(1, activation='tanh', kernel_initializer=glorot_init, use_bias=False)


        # Trainable inflow scaler
        # Irrigation branch
        self.inflow_dense1 = tf.keras.layers.Dense(neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.inflow_dense2 = tf.keras.layers.Dense(neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.inflow_dense3 = tf.keras.layers.Dense(neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.inflow_out = tf.keras.layers.Dense(1, activation='mish', kernel_initializer=scaled_he_init, use_bias=False)

        # Trainable inflow scaler
        # rain branch
        self.rain1_dense1 = tf.keras.layers.Dense(neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.rain1_dense2 = tf.keras.layers.Dense(neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.rain1_dense3 = tf.keras.layers.Dense(neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.rain1_out = tf.keras.layers.Dense(1, activation='mish', kernel_initializer=scaled_he_init, use_bias=False)

        self.rain2_dense1 = tf.keras.layers.Dense(neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.rain2_dense2 = tf.keras.layers.Dense(neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.rain2_dense3 = tf.keras.layers.Dense(neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.rain2_out = tf.keras.layers.Dense(1, activation='mish', kernel_initializer=scaled_he_init, use_bias=False)

        self.rain3_dense1 = tf.keras.layers.Dense(neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.rain3_dense2 = tf.keras.layers.Dense(neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.rain3_dense3 = tf.keras.layers.Dense(neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.rain3_out = tf.keras.layers.Dense(1, activation='mish', kernel_initializer=scaled_he_init, use_bias=False)

        self.rain4_dense1 = tf.keras.layers.Dense(neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.rain4_dense2 = tf.keras.layers.Dense(neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.rain4_dense3 = tf.keras.layers.Dense(neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.rain4_out = tf.keras.layers.Dense(1, activation='mish', kernel_initializer=scaled_he_init, use_bias=False)

        self.rain5_dense1 = tf.keras.layers.Dense(neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.rain5_dense2 = tf.keras.layers.Dense(neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.rain5_dense3 = tf.keras.layers.Dense(neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.rain5_out = tf.keras.layers.Dense(1, activation='mish', kernel_initializer=scaled_he_init, use_bias=False)

        self.rain6_dense1 = tf.keras.layers.Dense(neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.rain6_dense2 = tf.keras.layers.Dense(neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.rain6_dense3 = tf.keras.layers.Dense(neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.rain6_out = tf.keras.layers.Dense(1, activation='mish', kernel_initializer=scaled_he_init, use_bias=False)

        # plant_coef_1
        self.pc1_dense1 = tf.keras.layers.Dense(2*neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.pc1_dense2 = tf.keras.layers.Dense(2*neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.pc1_dense3 = tf.keras.layers.Dense(2*neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.pc1_dense4 = tf.keras.layers.Dense(2*neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.pc1_out = tf.keras.layers.Dense(1, activation='mish', kernel_initializer=glorot_init, use_bias=False)

        # plant_coef_2
        self.pc2_dense1 = tf.keras.layers.Dense(2*neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.pc2_dense2 = tf.keras.layers.Dense(2*neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.pc2_dense3 = tf.keras.layers.Dense(2*neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.pc2_dense4 = tf.keras.layers.Dense(2*neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.pc2_out = tf.keras.layers.Dense(1, activation='mish', kernel_initializer=glorot_init, use_bias=False)

        self.pc3_dense1 = tf.keras.layers.Dense(2*neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.pc3_dense2 = tf.keras.layers.Dense(2*neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.pc3_dense3 = tf.keras.layers.Dense(2*neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.pc3_dense4 = tf.keras.layers.Dense(2*neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.pc3_out = tf.keras.layers.Dense(1, activation='mish', kernel_initializer=glorot_init, use_bias=False)

        self.pc4_dense1 = tf.keras.layers.Dense(2*neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.pc4_dense2 = tf.keras.layers.Dense(2*neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.pc4_dense3 = tf.keras.layers.Dense(2*neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.pc4_dense4 = tf.keras.layers.Dense(2*neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.pc4_out = tf.keras.layers.Dense(1, activation='mish', kernel_initializer=glorot_init, use_bias=False)

        self.pc5_dense1 = tf.keras.layers.Dense(2*neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.pc5_dense2 = tf.keras.layers.Dense(2*neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.pc5_dense3 = tf.keras.layers.Dense(2*neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.pc5_dense4 = tf.keras.layers.Dense(2*neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.pc5_out = tf.keras.layers.Dense(1, activation='mish', kernel_initializer=glorot_init, use_bias=False)

        self.pc6_dense1 = tf.keras.layers.Dense(2*neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.pc6_dense2 = tf.keras.layers.Dense(2*neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.pc6_dense3 = tf.keras.layers.Dense(2*neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.pc6_dense4 = tf.keras.layers.Dense(2*neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.pc6_out = tf.keras.layers.Dense(1, activation='mish', kernel_initializer=glorot_init, use_bias=False)


        # Lateral outflow branches
        self.fl1_dense1 = tf.keras.layers.Dense(neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.fl1_dense2 = tf.keras.layers.Dense(neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.fl1_dense3 = tf.keras.layers.Dense(neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.fl1_out = tf.keras.layers.Dense(1, activation='mish', kernel_initializer=scaled_he_init, use_bias=False)

        self.fl2_dense1 = tf.keras.layers.Dense(neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.fl2_dense2 = tf.keras.layers.Dense(neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.fl2_dense3 = tf.keras.layers.Dense(neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.fl2_out = tf.keras.layers.Dense(1, activation='mish', kernel_initializer=scaled_he_init, use_bias=False)

        self.fl3_dense1 = tf.keras.layers.Dense(neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.fl3_dense2 = tf.keras.layers.Dense(neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.fl3_dense3 = tf.keras.layers.Dense(neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.fl3_out = tf.keras.layers.Dense(1, activation='mish', kernel_initializer=scaled_he_init, use_bias=False)

        self.fl4_dense1 = tf.keras.layers.Dense(neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.fl4_dense2 = tf.keras.layers.Dense(neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.fl4_dense3 = tf.keras.layers.Dense(neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.fl4_out = tf.keras.layers.Dense(1, activation='mish', kernel_initializer=scaled_he_init, use_bias=False)

        self.fl5_dense1 = tf.keras.layers.Dense(neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.fl5_dense2 = tf.keras.layers.Dense(neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.fl5_dense3 = tf.keras.layers.Dense(neurons, activation='mish', kernel_initializer=glorot_init, use_bias=False)
        self.fl5_out = tf.keras.layers.Dense(1, activation='mish', kernel_initializer=scaled_he_init, use_bias=False)



    def call(self, m, input_vec, t):

        # Unpack states
        m1 = tf.reshape(m[:, 0], [-1, 1])
        m2 = tf.reshape(m[:, 1], [-1, 1])
        m3 = tf.reshape(m[:, 2], [-1, 1])
        m4 = tf.reshape(m[:, 3], [-1, 1])
        m5 = tf.reshape(m[:, 4], [-1, 1])
        m6 = tf.reshape(m[:, 5], [-1, 1])

        # Compute differences between layers
        dm12 = m1 - m2
        dm23 = m2 - m3
        dm34 = m3 - m4
        dm45 = m4 - m5
        dm56 = m5 - m6

        # Input unpack
        qi   = tf.reshape(input_vec[:, 0], [-1, 1])  # irrigation
        rain = tf.reshape(input_vec[:, 1], [-1, 1])  # rain
        et0  = tf.reshape(input_vec[:, 2], [-1, 1])  # ET0
        t = tf.reshape(t, [-1, 1])  # time

        m1 = tf.identity(m1, name='m1')
        m2 = tf.identity(m2, name='m2')
        m3 = tf.identity(m3, name='m3')
        m4 = tf.identity(m4, name='m4')
        m5 = tf.identity(m5, name='m5')
        m6 = tf.identity(m6, name='m6')
        dm12 = tf.identity(dm12, name='dm12')
        dm23 = tf.identity(dm23, name='dm23')
        dm34 = tf.identity(dm34, name='dm34')
        dm45 = tf.identity(dm45, name='dm45')
        dm56 = tf.identity(dm56, name='dm56')
        qi = tf.identity(qi, name='qi')
        rain = tf.identity(rain, name='rain')
        et0 = tf.identity(et0, name='et0')
        t = tf.identity(t, name='t')



        # Outflow layer 1
        with tf.GradientTape() as tape:
            tape.watch(m1)
            x1 = self.f1_dense1(m1)
            x1 = self.f1_dense2(x1)
            x1 = self.f1_dense3(x1)
            f1 = self.f1_out(x1)
        df1_dm1 = tape.gradient(f1, m1)

        # Outflow layer 2
        with tf.GradientTape() as tape:
            tape.watch(m2)
            x2 = self.f2_dense1(m2)
            x2 = self.f2_dense2(x2)
            x2 = self.f2_dense3(x2)
            f2 = self.f2_out(x2)
        df2_dm2 = tape.gradient(f2, m2)

        # Outflow layer 3
        with tf.GradientTape() as tape:
            tape.watch(m3)
            x3 = self.f3_dense1(m3)
            x3 = self.f3_dense2(x3)
            x3 = self.f3_dense3(x3)
            f3 = self.f3_out(x3)
        df3_dm3 = tape.gradient(f3, m3)

        # Outflow layer 4
        with tf.GradientTape() as tape:
            tape.watch(m4)
            x4 = self.f4_dense1(m4)
            x4 = self.f4_dense2(x4)
            x4 = self.f4_dense3(x4)
            f4 = self.f4_out(x4)
        df4_dm4 = tape.gradient(f4, m4)

        # Outflow layer 5
        with tf.GradientTape() as tape:
            tape.watch(m5)
            x5 = self.f5_dense1(m5)
            x5 = self.f5_dense2(x5)
            x5 = self.f5_dense3(x5)
            f5 = self.f5_out(x5)
        df5_dm5 = tape.gradient(f5, m5)

        # Outflow layer 6
        with tf.GradientTape() as tape:
            tape.watch(m6)
            x6 = self.f6_dense1(m6)
            x6 = self.f6_dense2(x6)
            x6 = self.f6_dense3(x6)
            f6 = self.f6_out(x6)
        df6_dm6 = tape.gradient(f6, m6)

        # Interlayer f12
        with tf.GradientTape(persistent=True) as tape:
            tape.watch(m1)
            tape.watch(m2)
            m12_input = tf.concat([m1, m2], axis=1)
            x = self.f12_dense1(m12_input)
            x = self.f12_dense2(x)
            x = self.f12_dense3(x)
            f12 = self.f12_out(x)
        df12_dm1 = tape.gradient(f12, m1)
        df12_dm2 = tape.gradient(f12, m2)
        df12_ddm12 = df12_dm1 - df12_dm2

        with tf.GradientTape(persistent=True) as tape:
            tape.watch(m2)
            tape.watch(m3)
            m23_input = tf.concat([m2, m3], axis=1)
            x = self.f23_dense1(m23_input)
            x = self.f23_dense2(x)
            x = self.f23_dense3(x)
            f23 = self.f23_out(x)
        df23_dm2 = tape.gradient(f23, m2)
        df23_dm3 = tape.gradient(f23, m3)
        df23_ddm23 = df23_dm2 - df23_dm3

        with tf.GradientTape(persistent=True) as tape:
            tape.watch(m3)
            tape.watch(m4)
            m34_input = tf.concat([m3, m4], axis=1)
            x = self.f34_dense1(m34_input)
            x = self.f34_dense2(x)
            x = self.f34_dense3(x)
            f34 = self.f34_out(x)
        df34_dm3 = tape.gradient(f34, m3)
        df34_dm4 = tape.gradient(f34, m4)
        df34_ddm34 = df34_dm3 - df34_dm4

        with tf.GradientTape(persistent=True) as tape:
            tape.watch(m4)
            tape.watch(m5)
            m45_input = tf.concat([m4, m5], axis=1)
            x = self.f45_dense1(m45_input)
            x = self.f45_dense2(x)
            x = self.f45_dense3(x)
            f45 = self.f45_out(x)
        df45_dm4 = tape.gradient(f45, m4)
        df45_dm5 = tape.gradient(f45, m5)
        df45_ddm45 = df45_dm4 - df45_dm5

        with tf.GradientTape(persistent=True) as tape:
            tape.watch(m5)
            tape.watch(m6)
            m56_input = tf.concat([m5, m6], axis=1)
            x = self.f56_dense1(m56_input)
            x = self.f56_dense2(x)
            x = self.f56_dense3(x)
            f56 = self.f56_out(x)
        df56_dm5 = tape.gradient(f56, m5)
        df56_dm6 = tape.gradient(f56, m6)
        df56_ddm56 = df56_dm5 - df56_dm6

        # Inflow
        with tf.GradientTape() as tape:
            tape.watch(qi)
            x = self.inflow_dense1(qi)
            x = self.inflow_dense2(x)
            x = self.inflow_dense3(x)
            inflow = self.inflow_out(x)
        dinflow_dq = tape.gradient(inflow, qi)

        # Rain
        with tf.GradientTape() as tape:
            tape.watch(rain)
            x = self.rain1_dense1(rain)
            x = self.rain1_dense2(x)
            x = self.rain1_dense3(x)
            in_rain1 = self.rain1_out(x)
        din_rain1_dq = tape.gradient(in_rain1, rain)

        with tf.GradientTape() as tape:
            tape.watch(rain)
            x = self.rain2_dense1(rain)
            x = self.rain2_dense2(x)
            x = self.rain2_dense3(x)
            in_rain2 = self.rain2_out(x)
        din_rain2_dq = tape.gradient(in_rain2, rain)

        with tf.GradientTape() as tape:
            tape.watch(rain)
            x = self.rain3_dense1(rain)
            x = self.rain3_dense2(x)
            x = self.rain3_dense3(x)
            in_rain3 = self.rain3_out(x)
        din_rain3_dq = tape.gradient(in_rain3, rain)

        with tf.GradientTape() as tape:
            tape.watch(rain)
            x = self.rain4_dense1(rain)
            x = self.rain4_dense2(x)
            x = self.rain4_dense3(x)
            in_rain4 = self.rain4_out(x)
        din_rain4_dq = tape.gradient(in_rain4, rain)

        with tf.GradientTape() as tape:
            tape.watch(rain)
            x = self.rain5_dense1(rain)
            x = self.rain5_dense2(x)
            x = self.rain5_dense3(x)
            in_rain5 = self.rain5_out(x)
        din_rain5_dq = tape.gradient(in_rain5, rain)

        with tf.GradientTape() as tape:
            tape.watch(rain)
            x = self.rain6_dense1(rain)
            x = self.rain6_dense2(x)
            x = self.rain6_dense3(x)
            in_rain6 = self.rain6_out(x)
        din_rain6_dq = tape.gradient(in_rain6, rain)




        # Plant coef 1
                # Plant coef 1
        inputs_pc1 = tf.concat([m1, et0], axis=1)
        with tf.GradientTape() as tape:
            tape.watch(inputs_pc1)
            x_pc1 = self.pc1_dense1(inputs_pc1)
            x_pc1 = self.pc1_dense2(x_pc1)
            x_pc1 = self.pc1_dense3(x_pc1)
            x_pc1 = self.pc1_dense4(x_pc1)
            pc1 = self.pc1_out(x_pc1)
        dpc1_dinputs = tape.gradient(pc1, inputs_pc1)
        dpc1_dm1 = dpc1_dinputs[:, 0:1]
        dpc1_det0 = dpc1_dinputs[:, 1:2]

        # Plant coef 2
        inputs_pc2 = tf.concat([m2, et0], axis=1)
        with tf.GradientTape() as tape:
            tape.watch(inputs_pc2)
            x_pc2 = self.pc2_dense1(inputs_pc2)
            x_pc2 = self.pc2_dense2(x_pc2)
            x_pc2 = self.pc2_dense3(x_pc2)
            x_pc2 = self.pc2_dense4(x_pc2)
            pc2 = self.pc2_out(x_pc2)
        dpc2_dinputs = tape.gradient(pc2, inputs_pc2)
        dpc2_dm2 = dpc2_dinputs[:, 0:1]
        dpc2_det0 = dpc2_dinputs[:, 1:2]

        # Plant coef 3
        inputs_pc3 = tf.concat([m3, et0], axis=1)
        with tf.GradientTape() as tape:
            tape.watch(inputs_pc3)
            x_pc3 = self.pc3_dense1(inputs_pc3)
            x_pc3 = self.pc3_dense2(x_pc3)
            x_pc3 = self.pc3_dense3(x_pc3)
            x_pc3 = self.pc3_dense4(x_pc3)
            pc3 = self.pc3_out(x_pc3)
        dpc3_dinputs = tape.gradient(pc3, inputs_pc3)
        dpc3_dm3 = dpc3_dinputs[:, 0:1]
        dpc3_det0 = dpc3_dinputs[:, 1:2]

        # Plant coef 4
        inputs_pc4 = tf.concat([m4, et0], axis=1)
        with tf.GradientTape() as tape:
            tape.watch(inputs_pc4)
            x_pc4 = self.pc4_dense1(inputs_pc4)
            x_pc4 = self.pc4_dense2(x_pc4)
            x_pc4 = self.pc4_dense3(x_pc4)
            x_pc4 = self.pc4_dense4(x_pc4)
            pc4 = self.pc4_out(x_pc4)
        dpc4_dinputs = tape.gradient(pc4, inputs_pc4)
        dpc4_dm4 = dpc4_dinputs[:, 0:1]
        dpc4_det0 = dpc4_dinputs[:, 1:2]

        # Plant coef 5
        inputs_pc5 = tf.concat([m5, et0], axis=1)
        with tf.GradientTape() as tape:
            tape.watch(inputs_pc5)
            x_pc5 = self.pc5_dense1(inputs_pc5)
            x_pc5 = self.pc5_dense2(x_pc5)
            x_pc5 = self.pc5_dense3(x_pc5)
            x_pc5 = self.pc5_dense4(x_pc5)
            pc5 = self.pc5_out(x_pc5)
        dpc5_dinputs = tape.gradient(pc5, inputs_pc5)
        dpc5_dm5 = dpc5_dinputs[:, 0:1]
        dpc5_det0 = dpc5_dinputs[:, 1:2]

        # Plant coef 6
        inputs_pc6 = tf.concat([m6, et0], axis=1)
        with tf.GradientTape() as tape:
            tape.watch(inputs_pc6)
            x_pc6 = self.pc6_dense1(inputs_pc6)
            x_pc6 = self.pc6_dense2(x_pc6)
            x_pc6 = self.pc6_dense3(x_pc6)
            x_pc6 = self.pc6_dense4(x_pc6)
            pc6 = self.pc6_out(x_pc6)
        dpc6_dinputs = tape.gradient(pc6, inputs_pc6)
        dpc6_dm6 = dpc6_dinputs[:, 0:1]
        dpc6_det0 = dpc6_dinputs[:, 1:2]



        # Lateral outflows
        with tf.GradientTape() as tape:
            tape.watch(m1)
            x_fl1 = self.fl1_dense1(m1)
            x_fl1 = self.fl1_dense2(x_fl1)
            x_fl1 = self.fl1_dense3(x_fl1)
            fl1 = self.fl1_out(x_fl1)
        dfl1_dm1 = tape.gradient(fl1, m1)

        with tf.GradientTape() as tape:
            tape.watch(m2)
            x_fl2 = self.fl2_dense1(m2)
            x_fl2 = self.fl2_dense2(x_fl2)
            x_fl2 = self.fl2_dense3(x_fl2)
            fl2 = self.fl2_out(x_fl2)
        dfl2_dm2 = tape.gradient(fl2, m2)

        with tf.GradientTape() as tape:
            tape.watch(m3)
            x_fl3 = self.fl3_dense1(m3)
            x_fl3 = self.fl3_dense2(x_fl3)
            x_fl3 = self.fl3_dense3(x_fl3)
            fl3 = self.fl3_out(x_fl3)
        dfl3_dm3 = tape.gradient(fl3, m3)

        with tf.GradientTape() as tape:
            tape.watch(m4)
            x_fl4 = self.fl4_dense1(m4)
            x_fl4 = self.fl4_dense2(x_fl4)
            x_fl4 = self.fl4_dense3(x_fl4)
            fl4 = self.fl4_out(x_fl4)
        dfl4_dm4 = tape.gradient(fl4, m4)

        with tf.GradientTape() as tape:
            tape.watch(m5)
            x_fl5 = self.fl5_dense1(m5)
            x_fl5 = self.fl5_dense2(x_fl5)
            x_fl5 = self.fl5_dense3(x_fl5)
            fl5 = self.fl5_out(x_fl5)
        dfl5_dm5 = tape.gradient(fl5, m5)



        return (
            f1, f2, f3, f4, f5, f6,
            f12, f23, f34, f45, f56,
            inflow,
            in_rain1, in_rain2, in_rain3, in_rain4, in_rain5, in_rain6,
            pc1, pc2, pc3, pc4, pc5, pc6,
            fl1, fl2, fl3, fl4, fl5,
            df1_dm1, df2_dm2, df3_dm3, df4_dm4, df5_dm5, df6_dm6,
            df12_ddm12, df23_ddm23, df34_ddm34, df45_ddm45, df56_ddm56,
            dinflow_dq,
            din_rain1_dq, din_rain2_dq, din_rain3_dq, din_rain4_dq, din_rain5_dq, din_rain6_dq,
            dpc1_dm1, dpc2_dm2, dpc3_dm3, dpc4_dm4, dpc5_dm5, dpc6_dm6,
            dpc1_det0, dpc2_det0, dpc3_det0, dpc4_det0, dpc5_det0, dpc6_det0,
            dfl1_dm1, dfl2_dm2, dfl3_dm3, dfl4_dm4, dfl5_dm5
        )

In [6]:
# Build and load PINN model
pinn_model = TwoTankPINN(neurons=56)

# Initialize model with dummy data
_dummy_m = tf.zeros((1, 6), dtype=tf.float64)
_dummy_u = tf.zeros((1, 3), dtype=tf.float64)
_dummy_t = tf.zeros((1, 1), dtype=tf.float64)
_ = pinn_model(_dummy_m, _dummy_u, _dummy_t)

# Load trained weights
WEIGHTS_FILE = os.path.join(WEIGHTS_DIR, "pinn_weights_H56_lr7.97e-05.weights.h5")
pinn_model.load_weights(WEIGHTS_FILE)

print(f"✓ Model loaded from {WEIGHTS_FILE}")

✓ Model loaded from ../weights/pinn_weights_H56_lr7.97e-05.weights.h5


## 5. PINN Evaluation via RK4 Integration
Run the trained PINN as an ODE system using 4th-order Runge-Kutta integration over training and forecast periods.

In [7]:
@tf.function
def system_func_pinn(t, state, input_vec):
    m_batch = tf.reshape(state, [1, 6])
    in_batch = tf.reshape(input_vec, [1, 3])
    t_batch = tf.reshape(t, [1, 1])

    # Updated unpacking to include six rain inputs
    (f1, f2, f3, f4, f5, f6,
     f12, f23, f34, f45, f56,
     inflow,
     in_rain1, in_rain2, in_rain3, in_rain4, in_rain5, in_rain6,
     pc1, pc2, pc3, pc4, pc5, pc6,
     fl1, fl2, fl3, fl4, fl5,
     *_) = pinn_model(m_batch, in_batch, t_batch)

    # Drop batch dimension
    f1, f2, f3, f4, f5, f6 = f1[0, 0], f2[0, 0], f3[0, 0], f4[0, 0], f5[0, 0], f6[0, 0]
    f12, f23, f34, f45, f56 = f12[0, 0], f23[0, 0], f34[0, 0], f45[0, 0], f56[0, 0]
    inflow = inflow[0, 0]
    in_rain1, in_rain2, in_rain3 = in_rain1[0, 0], in_rain2[0, 0], in_rain3[0, 0]
    in_rain4, in_rain5, in_rain6 = in_rain4[0, 0], in_rain5[0, 0], in_rain6[0, 0]
    pc1, pc2, pc3 = pc1[0, 0], pc2[0, 0], pc3[0, 0]
    pc4, pc5, pc6 = pc4[0, 0], pc5[0, 0], pc6[0, 0]
    fl1, fl2, fl3 = fl1[0, 0], fl2[0, 0], fl3[0, 0]
    fl4, fl5 = fl4[0, 0], fl5[0, 0]

    # Updated mass balances with rain per layer
    dh1dt = inflow - f1 - f12 - pc1 + in_rain1 - fl1
    dh2dt = f1 + f12 - f2 - f23 - pc2 + in_rain2 - fl2
    dh3dt = f2 + f23 - f3 - f34 - pc3 + in_rain3 - fl3
    dh4dt = f3 + f34 - f4 - f45 - pc4 + in_rain4 - fl4
    dh5dt = f4 + f45 - f5 - f56 - pc5 + in_rain5 - fl5
    dh6dt = f5 + f56 - f6 - pc6 + in_rain6

    return tf.stack([dh1dt, dh2dt, dh3dt, dh4dt, dh5dt, dh6dt])

In [8]:
# Run RK4 integration for training data
dt = 1.0
m_pinn_train_tf = rk4_integration(system_func_pinn, t_train_tf, m0_train_np, dt, forced_data=in_mid_train)
m_pinn_train_np = m_pinn_train_tf.numpy()
m_pinn_train_np_denorm = m_pinn_train_np * normalize["moisture"]

print(f"✓ Training integration complete: shape {m_pinn_train_np.shape}, range [{m_pinn_train_np_denorm.min():.1f}, {m_pinn_train_np_denorm.max():.1f}] %VWC")

2026-04-04 13:59:36.168022: W tensorflow/core/util/util.cc:163] Not handling type DT_DOUBLE
2026-04-04 13:59:36.168099: W tensorflow/core/util/util.cc:163] Not handling type DT_DOUBLE
2026-04-04 13:59:36.168186: W tensorflow/core/util/util.cc:163] Not handling type DT_DOUBLE
2026-04-04 13:59:36.168234: W tensorflow/core/util/util.cc:163] Not handling type DT_DOUBLE
2026-04-04 13:59:36.168304: W tensorflow/core/util/util.cc:163] Not handling type DT_DOUBLE
2026-04-04 13:59:36.168339: W tensorflow/core/util/util.cc:163] Not handling type DT_DOUBLE
2026-04-04 13:59:36.168400: W tensorflow/core/util/util.cc:163] Not handling type DT_DOUBLE
2026-04-04 13:59:36.168431: W tensorflow/core/util/util.cc:163] Not handling type DT_DOUBLE
2026-04-04 13:59:36.179932: W tensorflow/core/util/util.cc:163] Not handling type DT_DOUBLE
2026-04-04 13:59:36.179995: W tensorflow/core/util/util.cc:163] Not handling type DT_DOUBLE
2026-04-04 13:59:36.180056: W tensorflow/core/util/util.cc:163] Not handling typ

✓ Training integration complete: shape (840, 6), range [2.3, 18.1] %VWC


In [9]:
# Prepare validation data for PINN evaluation
# Apply normalization
for col in ["time_hours", "rain", "et0", "irrigation"]:
    df_val[f"{col}_scaled"] = df_val[col] / normalize.get(col.replace("_hours", ""), normalize[col])

for i in range(1, 7):
    df_val[f"moisture_{i}_scaled"] = df_val[f"moisture_{i}"] / normalize["moisture"]

df_val = df_val.sort_values("time_hours").dropna().reset_index(drop=True)

# Create tensors
t_val_tf = tf.constant(df_val["time_hours_scaled"].values.astype(np.float64), dtype=tf.float64)
inputs_val_tf = tf.constant(df_val[["irrigation_scaled", "rain_scaled", "et0_scaled"]].values.astype(np.float64), dtype=tf.float64)
states_val_tf = tf.constant(df_val[[f"moisture_{i}_scaled" for i in range(1, 7)]].values.astype(np.float64), dtype=tf.float64)

# Extract numpy arrays and initial conditions
m0_val_np = states_val_tf[0].numpy()

# Mid-step values for measured data
m_mid_val = states_val_tf[:-1]
in_mid_val = inputs_val_tf[:-1]
m_mid_val_np_denorm = m_mid_val.numpy() * normalize["moisture"]

# Run RK4 integration for validation data
m_pinn_val_tf = rk4_integration(system_func_pinn, t_val_tf, m0_val_np, dt, forced_data=in_mid_val)
m_pinn_val_np = m_pinn_val_tf.numpy()
m_pinn_val_np_denorm = m_pinn_val_np * normalize["moisture"]

print(f"✓ Validation data prepared: {len(t_val_tf)} time points")
print(f"✓ Validation integration complete: shape {m_pinn_val_np.shape}, range [{m_pinn_val_np_denorm.min():.1f}, {m_pinn_val_np_denorm.max():.1f}] %VWC")

2026-04-04 13:59:39.904135: W tensorflow/core/util/util.cc:163] Not handling type DT_DOUBLE
2026-04-04 13:59:39.904205: W tensorflow/core/util/util.cc:163] Not handling type DT_DOUBLE
2026-04-04 13:59:39.904280: W tensorflow/core/util/util.cc:163] Not handling type DT_DOUBLE
2026-04-04 13:59:39.904309: W tensorflow/core/util/util.cc:163] Not handling type DT_DOUBLE
2026-04-04 13:59:39.904362: W tensorflow/core/util/util.cc:163] Not handling type DT_DOUBLE
2026-04-04 13:59:39.904391: W tensorflow/core/util/util.cc:163] Not handling type DT_DOUBLE
2026-04-04 13:59:39.904443: W tensorflow/core/util/util.cc:163] Not handling type DT_DOUBLE
2026-04-04 13:59:39.904471: W tensorflow/core/util/util.cc:163] Not handling type DT_DOUBLE
2026-04-04 13:59:39.915266: W tensorflow/core/util/util.cc:163] Not handling type DT_DOUBLE
2026-04-04 13:59:39.915312: W tensorflow/core/util/util.cc:163] Not handling type DT_DOUBLE
2026-04-04 13:59:39.915354: W tensorflow/core/util/util.cc:163] Not handling typ

✓ Validation data prepared: 73 time points
✓ Validation integration complete: shape (73, 6), range [3.6, 15.1] %VWC


## 6. Figure 1 — Training and Forecast Evaluation
Generates `estimation_vs_validation_with_plant_uptake.pdf`

Side-by-side heatmap comparing measured and PINN-estimated soil moisture (%VWC at depths 5–55 cm) over the training and 3-day forecast periods, together with external forcings (rain, irrigation, ET₀) and estimated plant water uptake at each depth.

In [10]:
# Visualization configuration
BAR_COLORS = {"rain": "rgb(0, 0, 180)", "irrigation": "rgb(0, 0, 250)", "et0": "rgb(255, 69, 0)"}
INPUT_LABELS = {"rain": "Rain (mm/h)", "irrigation": "Irrigation q(h/h)", "et0": "ET₀ (mm/h)"}
MOISTURE_RANGE = (3.5, 10.2)
PLANT_SYMBOLS = ["pc₁", "pc₂", "pc₃", "pc₄", "pc₅", "pc₆"]
PLANT_DEPTHS = ["5cm", "15cm", "25cm", "35cm", "45cm", "55cm"]
PLANT_COLOR = "rgb(255, 69, 0)"
COLOR_SCALE = [[0.0, "rgb(250, 70, 22)"], [1.0, "rgb(0, 33, 165)"]]

def normalize_moisture(data, min_vwc, max_vwc):
    return np.clip((data - min_vwc) / (max_vwc - min_vwc), 0.0, 1.0)

# Prepare plot data
plot_train = {
    "time": df_train["timestamp"].to_numpy(),
    "inputs": df_train[["rain", "irrigation", "et0"]],
    "measured_heatmap": normalize_moisture(m_mid_train_np_denorm, *MOISTURE_RANGE).T,
    "predicted_heatmap": normalize_moisture(m_pinn_train_np_denorm, *MOISTURE_RANGE).T,
}

plot_val = {
    "time": df_val["timestamp"].to_numpy(),
    "inputs": df_val[["rain", "irrigation", "et0"]],
    "measured_heatmap": normalize_moisture(m_mid_val_np_denorm, *MOISTURE_RANGE).T,
    "predicted_heatmap": normalize_moisture(m_pinn_val_np_denorm, *MOISTURE_RANGE).T,
}

plot_meta = {"column_widths": [width_train, width_val]}

print(f"✓ Visualization data ready: {len(plot_train['time'])} training + {len(plot_val['time'])} validation points")

✓ Visualization data ready: 840 training + 73 validation points


In [11]:
# Extract plant uptake components
def extract_plant_uptake(states_mid, inputs_mid, times_mid):
    outputs = pinn_model(states_mid, inputs_mid, times_mid)
    # Extract pc1-pc6 (indices 18-23 in model output)
    pcs = outputs[18:24]
    return [pc.numpy().ravel() * normalize["moisture"] for pc in pcs]

t_mid_train = t_train_tf[:-1]
t_mid_val = t_val_tf[:-1]

plant_uptake_train = extract_plant_uptake(m_mid_train, in_mid_train, t_mid_train)
plant_uptake_val = extract_plant_uptake(m_mid_val, in_mid_val, t_mid_val)

print(f"✓ Plant uptake extracted: 6 layers for training and validation")

✓ Plant uptake extracted: 6 layers for training and validation


In [12]:
# Build side-by-side figure
def build_figure(train, val, plant_train, plant_val, meta):
    """
    Create side-by-side comparison figure with 11 rows:
    - Rows 1-3: Input forcings (rain, irrigation, ET0)
    - Row 4: Measured soil moisture heatmap
    - Row 5: PINN predicted soil moisture heatmap
    - Rows 6-11: Plant uptake for each layer (pc1-pc6)
    """
    # Font size multiplier - change this value to scale all fonts proportionally
    FONT_SCALE = 3.5  # 1.0 = default, 1.2 = 20% larger, 0.8 = 20% smaller

    column_widths = meta["column_widths"]
    
    # ENHANCED: Increased height by 50% - keeping ORIGINAL proportions for row heights
    # Original: [0.06, 0.06, 0.06, 0.18, 0.18, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05]
    row_heights = [0.06, 0.06, 0.06, 0.18, 0.18, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05]

    # Define subplot types: 3 bar rows + 2 heatmap rows + 6 bar rows
    specs = [[{"type": "bar"}, {"type": "bar"}] for _ in range(3)]
    specs += [[{"type": "heatmap"}, {"type": "heatmap"}] for _ in range(2)]
    specs += [[{"type": "bar"}, {"type": "bar"}] for _ in range(6)]

    fig = make_subplots(
        rows=11, cols=2, row_heights=row_heights, column_widths=column_widths,
        shared_xaxes="columns", vertical_spacing=0.02, horizontal_spacing=0.035, specs=specs
    )

    # === ROWS 1-3: Input forcings (rain, irrigation, et0) ===
    # Simple names without units for left side, only units for right side
    INPUT_NAMES = {"rain": "Rain", "irrigation": "Irrigation", "et0": "ET₀"}
    INPUT_UNITS = {"rain": "mm/h", "irrigation": "h/h", "et0": "mm/h"}
    
    # FIXED: Clean tick values for each input type
    INPUT_TICKVALS = {
        "rain": [0, 10, 20],
        "irrigation": [0, 0.5, 1],
        "et0": [0, 0.25, 0.5]
    }
    INPUT_TICKTEXT = {
        "rain": ["0", "10", "20"],
        "irrigation": ["0", "0.5", "1"],
        "et0": ["0", "0.25", "0.5"]
    }

    for row_idx, key in enumerate(["rain", "irrigation", "et0"], start=1):
        label_simple = INPUT_NAMES[key]  # Simple name (e.g., "Rain")
        label_units = INPUT_UNITS[key]   # Only units (e.g., "(mm/h)")

        # Training column
        fig.add_trace(go.Bar(
            x=train["time"], y=train["inputs"][key], name=label_simple,
            marker=dict(color=BAR_COLORS[key], line=dict(width=0)),
            showlegend=False  # Hide legend for cleaner appearance
        ), row=row_idx, col=1)
        # Validation column
        fig.add_trace(go.Bar(
            x=val["time"], y=val["inputs"][key], name=label_simple,
            marker=dict(color=BAR_COLORS[key], line=dict(width=0)),
            showlegend=False
        ), row=row_idx, col=2)

        # Calculate shared y-axis range for homogeneous display
        all_values = np.concatenate([train["inputs"][key], val["inputs"][key]])
        y_max = np.max(all_values)
        y_range = [0, y_max * 1.05]  # Add 5% padding
        
        # Use fixed clean tick values
        input_tickvals = INPUT_TICKVALS[key]
        input_ticktext = INPUT_TICKTEXT[key]

        # Training column (col 1): Show name label, consistent gridlines with Forecast
        fig.update_yaxes(
            title=dict(text=label_simple, font=dict(size=int(24*FONT_SCALE))),
            showticklabels=False,  # Hide tick values on left
            tickvals=input_tickvals,  # Same tick positions as Forecast
            showgrid=True,  # Show gridlines
            gridcolor="lightgray",
            gridwidth=1,
            range=y_range,
            row=row_idx, col=1
        )
        # Validation column (col 2): Show tick values on RIGHT side, same gridlines
        fig.update_yaxes(
            title=dict(text=label_units, font=dict(size=int(24*FONT_SCALE))),
            side="right",
            tickfont=dict(size=int(20*FONT_SCALE)),
            tickvals=input_tickvals,  # Same tick positions as Training
            ticktext=input_ticktext,  # Clean labels
            showgrid=True,  # Show gridlines
            gridcolor="lightgray",
            gridwidth=1,
            range=y_range,
            row=row_idx, col=2
        )

    # === ROWS 4-5: Soil moisture heatmaps (measured + predicted) ===
    for col_idx, dataset in enumerate([train, val], start=1):
        # Row 4: Measured
        fig.add_trace(go.Heatmap(
            x=dataset["time"], y=np.arange(1, 7), z=dataset["measured_heatmap"],
            colorscale=COLOR_SCALE, showscale=False, zmin=0.0, zmax=1.0
        ), row=4, col=col_idx)

        # Row 5: Predicted
        fig.add_trace(go.Heatmap(
            x=dataset["time"], y=np.arange(1, 7), z=dataset["predicted_heatmap"],
            colorscale=COLOR_SCALE, showscale=(col_idx == 2),
            colorbar=dict(
                title=dict(text="%VWC", font=dict(size=int(24*FONT_SCALE))),
                tickvals=[0.0, 0.5, 1.0],
                ticktext=[f"{MOISTURE_RANGE[0]:.1f}%", f"{sum(MOISTURE_RANGE)/2:.1f}%", f"{MOISTURE_RANGE[1]:.1f}%"],
                # RESTORED: Original colorbar settings matching rows 4-5
                # row_heights[3] + row_heights[4] = 0.18 + 0.18 = 0.36
                len=0.36,
                # Position at the vertical center of rows 4-5
                # In Plotly coords (y=1 is top): rows 1-3 take 0.18, so row 4 starts at 0.82
                # Rows 4-5 span 0.36, so they go from 0.82 to 0.46, center = 0.64
                y=0.6,   # Vertical center of the two heatmap rows
                yanchor="middle",
                x=1.0,
                xanchor="left",
                tickfont=dict(size=int(20*FONT_SCALE))
            ) if col_idx == 2 else None,
            zmin=0.0, zmax=1.0
        ), row=5, col=col_idx)

    # Y-axis labels for heatmaps
    for row, title in [(4, "Measured SM"), (5, "Estimated SM")]:
        fig.update_yaxes(
            title=dict(text=title, font=dict(size=int(24*FONT_SCALE))), tickvals=np.arange(1, 7),
            ticktext=PLANT_DEPTHS, autorange="reversed", tickfont=dict(size=int(18*FONT_SCALE)), row=row, col=1
        )
        fig.update_yaxes(autorange="reversed", showticklabels=False, row=row, col=2)

    # === ROWS 6-11: Plant uptake for each layer ===
    time_train_pc = train["time"][:-1]
    time_val_pc = val["time"][:-1]

    # Calculate shared y-axis range
    all_series = [*plant_train, *plant_val]
    y_min = min(float(np.min(arr)) for arr in all_series)
    y_max = max(float(np.max(arr)) for arr in all_series)
    y_range = [y_min - 0.05 * (y_max - y_min), y_max + 0.05 * (y_max - y_min)]
    
    # Define consistent tick values for plant uptake (same for both columns)
    plant_tickvals = [0.0, 0.1, 0.2]

    for idx, (depth, series_train, series_val) in enumerate(zip(PLANT_DEPTHS, plant_train, plant_val), start=6):
        # Training column
        fig.add_trace(go.Bar(
            x=time_train_pc, y=series_train, name=depth,
            marker=dict(color=PLANT_COLOR, line=dict(width=0)),
            showlegend=False
        ), row=idx, col=1)
        # Validation column
        fig.add_trace(go.Bar(
            x=time_val_pc, y=series_val, name=depth,
            marker=dict(color=PLANT_COLOR, line=dict(width=0)),
            showlegend=False
        ), row=idx, col=2)

        # Training column (col 1): Show depth label, same gridlines as Forecast
        fig.update_yaxes(
            title=dict(text=depth, font=dict(size=int(20*FONT_SCALE))),
            showticklabels=False,  # Hide tick labels on left
            tickvals=plant_tickvals,  # Same tick positions as Forecast
            showgrid=True,  # Show gridlines
            gridcolor="lightgray",
            gridwidth=1,
            range=y_range,
            row=idx, col=1
        )
        # Validation column (col 2): Show tick values on RIGHT side
        fig.update_yaxes(
            side="right",
            tickfont=dict(size=int(18*FONT_SCALE)),
            tickvals=plant_tickvals,  # Same tick positions as Training
            ticktext=["0", "0.1", "0.2"],  # Clean labels
            showgrid=True,  # Show gridlines
            gridcolor="lightgray",
            gridwidth=1,
            range=y_range,
            row=idx, col=2
        )

    # === X-AXIS FORMATTING ===
    # Hide all x-axis labels except bottom row
    for row in range(1, 11):
        for col in [1, 2]:
            fig.update_xaxes(showticklabels=False, row=row, col=col)

    # Bottom row: show date labels
    # Training column (col 1): show ~10 ticks (dtick in milliseconds: 3 days = 3*24*60*60*1000)
    fig.update_xaxes(tickformat="%b %d", tickfont=dict(size=int(20*FONT_SCALE)), dtick=3*24*60*60*1000, row=11, col=1)
    # Validation column (col 2): keep default ticks
    fig.update_xaxes(tickformat="%b %d", tickfont=dict(size=int(20*FONT_SCALE)), row=11, col=2)

    # === LAYOUT ===
    # ENHANCED: Increased height from 2800 to 4200 (50% increase)
    fig.update_layout(
        template="plotly_white", width=4800, height=4200,
        font=dict(family="Arial", size=int(24*FONT_SCALE)), margin=dict(l=150, r=130, t=160, b=90),
        showlegend=False,  # Hide legend completely
        bargap=0.0
    )

    # Column titles - centered over each column
    # Training column center = left edge + half of training width
    training_center = column_widths[0] / 2
    # Validation column center = training width + half of validation width
    validation_center = column_widths[0] + (column_widths[1] / 2)

    fig.add_annotation(
        text="<b>Training</b>",
        xref="paper", yref="paper",
        x=training_center, y=1,
        xanchor="center", yanchor="bottom",
        showarrow=False, font=dict(size=int(30*FONT_SCALE))
    )
    fig.add_annotation(
        text="<b>Forecast</b>",
        xref="paper", yref="paper",
        x=validation_center, y=1,
        xanchor="center", yanchor="bottom",
        showarrow=False, font=dict(size=int(30*FONT_SCALE))
    )

    # Shared titles for plant uptake rows (rows 6-11)
    # Rows 1-5 take: 0.06*3 + 0.18*2 = 0.54 (54% from top)
    # Rows 6-11 take: 0.05*6 = 0.30 (30% from top, going from 54% to 84%)
    # So plant uptake rows span from y=0.46 to y=0.16 in Plotly coords
    # Center is at y = (0.46 + 0.16)/2 = 0.31

    # Left side: name only
    fig.add_annotation(
        text="Estimated plant uptake",
        xref="paper", yref="paper",
        x=-0.03, y=0.2,  # Left side, centered vertically on rows 6-11
        xanchor="right", yanchor="middle",
        showarrow=False, font=dict(size=int(24*FONT_SCALE)),
        textangle=-90  # Rotate text vertically
    )

    # Right side: units only
    fig.add_annotation(
        text="(%VWC/h)",
        xref="paper", yref="paper",
        x=1.02, y=0.2,  # Right side, centered vertically on rows 6-11
        xanchor="left", yanchor="middle",
        showarrow=False, font=dict(size=int(24*FONT_SCALE)),
        textangle=-90  # Rotate text vertically
    )

    return fig


# Generate and save figure
fig = build_figure(plot_train, plot_val, plant_uptake_train, plant_uptake_val, plot_meta)

# ENHANCED: Updated output dimensions - 50% taller (2800 -> 4200)
output_path = "estimation_vs_validation_with_plant_uptake.pdf"
fig.write_image(output_path, format="pdf", width=4800, height=4200, scale=2)

print(f"✓ Figure saved to {output_path}")
fig.show()

✓ Figure saved to estimation_vs_validation_with_plant_uptake.pdf


## 7. Table 1 and Figure 2 — Error Analysis
Generates `error_analysis_heatmap_boxplot.pdf` and prints **Table 1** statistics.

- **Heatmap**: signed estimation error (measured − predicted) over time at each depth
- **Boxplots**: per-layer error distributions for training and forecast
- **Table 1**: RMSE, standard deviation, and R² for each soil layer

In [13]:
# ============================================================================
# ERROR ANALYSIS: HEATMAPS + BOX PLOTS
# ============================================================================

# Font size multiplier - change this value to scale all fonts proportionally
FONT_SCALE = 3 # 1.0 = default, 1.2 = 20% larger, 0.8 = 20% smaller

width_train = 0.92
width_val = 0.08

# Create figure with 2 rows, 2 columns - PROPORTIONAL to data amount
# Row 1: Error heatmaps
# Row 2: Error box plots
fig_error_heatmap = make_subplots(
    rows=2, cols=2,
    row_heights=[0.70, 0.30],  # Heatmap bigger, box plots smaller
    subplot_titles=["<b>Training</b>", "<b>Forecast</b>", "", ""],
    column_widths=[width_train, width_val],  # Proportional: [0.92, 0.08]
    horizontal_spacing=0.02,  # Small gap between columns
    vertical_spacing=0.15,  # Gap between heatmap and box plots (increase for more space)
    specs=[
        [{"type": "heatmap"}, {"type": "heatmap"}],
        [{"type": "box"}, {"type": "box"}]
    ]
)

# Prepare training error data (time x layers) - SIGNED ERRORS (measured - predicted)
train_errors = m_mid_train_np_denorm - m_pinn_train_np_denorm[:-1]  # Shape: (839, 6)
train_errors_t = train_errors.T  # Transpose to (6, 839) for heatmap

# Prepare validation error data - SIGNED ERRORS (measured - predicted)
val_errors = m_mid_val_np_denorm - m_pinn_val_np_denorm[:-1]  # Shape: (72, 6)
val_errors_t = val_errors.T  # Transpose to (6, 72) for heatmap

# Calculate ACTUAL min/max errors from the data (ASYMMETRIC)
all_errors_combined = np.concatenate([train_errors.flatten(), val_errors.flatten()])
actual_min_error = all_errors_combined.min()
actual_max_error = all_errors_combined.max()

# Use ASYMMETRIC color scale based on actual data range
vmin = actual_min_error  # Actual minimum (predicted too high)
vmax = actual_max_error  # Actual maximum (predicted too low)

print(f"Error range for heatmap (measured - predicted):")
print(f"  Min error: {vmin:.3f} %VWC (negative = predicted too high)")
print(f"  Max error: {vmax:.3f} %VWC (positive = predicted too low)")

# Create custom colorscale with WHITE at ZERO (accounting for asymmetric range)
# Calculate where zero falls in the [vmin, vmax] range
zero_position = abs(vmin) / (abs(vmin) + abs(vmax))
print(f"  Zero position in colorscale: {zero_position:.3f}")

# Custom colorscale: Blue (negative) → White (zero) → Red (positive)
custom_colorscale = [
    [0.0, "rgb(5, 48, 97)"],           # Dark blue at vmin
    [zero_position * 0.5, "rgb(33, 102, 172)"],  # Medium blue
    [zero_position, "rgb(255, 255, 255)"],       # WHITE at ZERO
    [zero_position + (1 - zero_position) * 0.5, "rgb(239, 138, 98)"],  # Medium red
    [1.0, "rgb(178, 24, 43)"]          # Dark red at vmax
]

# Create datetime arrays for x-axis (use timestamp arrays from dataframes)
time_train_datetimes = df_train["timestamp"].to_numpy()[:-1]  # Mid-step times for training
time_val_datetimes = df_val["timestamp"].to_numpy()[:-1]  # Mid-step times for validation

# Add training heatmap
fig_error_heatmap.add_trace(
    go.Heatmap(
        z=train_errors_t,
        x=time_train_datetimes,  # Use datetime array
        y=[1, 2, 3, 4, 5, 6],  # Use numeric for y-axis
        colorscale=custom_colorscale,  # Custom scale with white at zero
        zmin=vmin,
        zmax=vmax,
        colorbar=dict(
            #title=dict(text="%VWC", font=dict(size=int(10*FONT_SCALE))),
            tickfont=dict(size=int(22*FONT_SCALE)),
            len=0.65,  # Match row 1 height (70% of figure)
            thickness=30,
            x=1,
            y=0.7,  # Vertical center of row 1 (heatmap occupies top 70%, so center is at ~85%)
            yanchor="middle",  # Anchor to middle like the successful figure above
            tickvals=[vmin, 0, vmax],
            ticktext=[f"{vmin:.1f}", "<b>0</b>", f"{vmax:.1f}"]
        ),
        hovertemplate='<b>Layer %{y}</b><br>Time: %{x}<br>Error: %{z:.2f} %VWC<extra></extra>'
    ),
    row=1, col=1
)

# Add validation heatmap
fig_error_heatmap.add_trace(
    go.Heatmap(
        z=val_errors_t,
        x=time_val_datetimes,  # Use datetime array
        y=[1, 2, 3, 4, 5, 6],  # Use numeric for y-axis
        colorscale=custom_colorscale,  # Custom scale with white at zero
        zmin=vmin,
        zmax=vmax,
        showscale=False,
        hovertemplate='<b>Layer %{y}</b><br>Time: %{x}<br>Error: %{z:.2f} %VWC<extra></extra>'
    ),
    row=1, col=2
)

# ============================================================================
# ROW 2: BOX PLOTS OF ERROR DISTRIBUTION (All layers on same scale)
# ============================================================================

# Add box plots for each layer - TRAINING
layer_names = ["5cm", "15cm", "25cm", "35cm", "45cm", "55cm"]
for i in range(6):
    fig_error_heatmap.add_trace(
        go.Box(
            y=train_errors[:, i],
            name=layer_names[i],
            marker=dict(
                color='rgba(31,119,180,0.7)',
                size=8  # Larger mean marker (diamond)
            ),
            boxmean='sd',  # Show mean (diamond) and standard deviation (dashed lines)
            line=dict(width=2.5, color='rgb(31,119,180)'),  # Thicker box outline and SD lines
            fillcolor='rgba(31,119,180,0.5)',
            showlegend=False
        ),
        row=2, col=1
    )

# Add box plots for each layer - VALIDATION
for i in range(6):
    fig_error_heatmap.add_trace(
        go.Box(
            y=val_errors[:, i],
            name=layer_names[i],
            marker=dict(
                color='rgba(255,127,14,0.7)',
                size=8  # Larger mean marker (diamond)
            ),
            boxmean='sd',  # Show mean (diamond) and standard deviation (dashed lines)
            line=dict(width=2.5, color='rgb(255,127,14)'),  # Thicker box outline and SD lines
            fillcolor='rgba(255,127,14,0.5)',
            showlegend=False
        ),
        row=2, col=2
    )

# Update layout - MATCH ENHANCED FIGURE STYLE
# ENHANCED: Increased height by 50% (1000 -> 1500)
fig_error_heatmap.update_layout(
    template="plotly_white",
    height=1500,  # 50% taller to match main figure enhancement
    width=4800,  # Same width as enhanced figure
    font=dict(family="Arial", size=int(28*FONT_SCALE)),
    margin=dict(l=180, r=140, t=120, b=80),
    bargap=0
)

# Update subplot titles - match enhanced figure style
fig_error_heatmap.update_annotations(font=dict(size=int(35*FONT_SCALE), family="Arial"))

# Y-axes - MATCH ENHANCED FIGURE HEATMAP STYLE
fig_error_heatmap.update_yaxes(
    title=dict(text="Error (%VWC)", font=dict(size=int(28*FONT_SCALE))),
    tickvals=np.arange(1, 7),
    ticktext=["5cm", "15cm", "25cm", "35cm", "45cm", "55cm"],
    autorange="reversed",  # Top to bottom like enhanced figure
    tickfont=dict(size=int(24*FONT_SCALE)),
    row=1, col=1
)

fig_error_heatmap.update_yaxes(
    title=None,
    tickvals=np.arange(1, 7),
    showticklabels=False,  # No labels on second column
    autorange="reversed",
    row=1, col=2
)

# X-axes for heatmaps
# Training column (col 1): normal tick display
fig_error_heatmap.update_xaxes(
    tickformat="%b %d",  # Show dates as "Jan 01"
    tickfont=dict(size=int(24*FONT_SCALE)),
    row=1, col=1
)
# Validation column (col 2): only ONE tick label (centered) for narrow column
fig_error_heatmap.update_xaxes(
    tickformat="%b %d",
    tickfont=dict(size=int(24*FONT_SCALE)),
    nticks=1,  # Show only 1 tick label
    row=1, col=2
)

# Y-axes for box plots - Gridlines at 2.5 intervals, but labels only at 5.0 intervals
# Calculate gridline values at 2.5 intervals for visual guides
tick_start = np.floor(vmin / 2.5) * 2.5
tick_end = np.ceil(vmax / 2.5) * 2.5
gridline_values = np.arange(tick_start, tick_end + 2.5, 2.5)

# Calculate label values at 5.0 intervals for cleaner appearance
label_start = np.floor(vmin / 5.0) * 5.0
label_end = np.ceil(vmax / 5.0) * 5.0
label_values = np.arange(label_start, label_end + 5.0, 5.0)

# Left column - show labels at 5.0 intervals
fig_error_heatmap.update_yaxes(
    title=dict(text="Distribution", font=dict(size=int(20*FONT_SCALE))),
    range=[vmin, vmax],
    tickfont=dict(size=int(24*FONT_SCALE)),
    tickmode='array',
    tickvals=label_values,  # Show labels only at 5.0 intervals (e.g., -5, 0, 5, 10)
    showgrid=True,
    gridwidth=1,
    gridcolor='lightgray',
    griddash='solid',
    zeroline=True,
    zerolinewidth=3,  # Thicker zero line
    zerolinecolor='black',
    row=2, col=1
)

# Add minor gridlines at 2.5 intervals by using layer='below grid'
fig_error_heatmap.update_yaxes(
    minor=dict(
        tickmode='array',
        tickvals=gridline_values,  # All gridlines at 2.5 intervals
        showgrid=True,
        gridwidth=1,
        gridcolor='rgba(211,211,211,0.5)'  # Lighter gray for minor gridlines
    ),
    row=2, col=1
)

# Right column - no labels, same gridlines
fig_error_heatmap.update_yaxes(
    title=None,
    range=[vmin, vmax],
    tickfont=dict(size=int(24*FONT_SCALE)),
    showticklabels=False,
    tickmode='array',
    tickvals=label_values,
    showgrid=True,
    gridwidth=1,
    gridcolor='lightgray',
    griddash='solid',
    zeroline=True,
    zerolinewidth=3,  # Thicker zero line
    zerolinecolor='black',
    row=2, col=2
)

# Add minor gridlines for right column too
fig_error_heatmap.update_yaxes(
    minor=dict(
        tickmode='array',
        tickvals=gridline_values,
        showgrid=True,
        gridwidth=1,
        gridcolor='rgba(211,211,211,0.5)'
    ),
    row=2, col=2
)

# X-axes for box plots
for col in [1, 2]:
    fig_error_heatmap.update_xaxes(
        tickvals=list(range(6)),
        ticktext=layer_names,
        tickfont=dict(size=int(20*FONT_SCALE)),
        row=2, col=col
    )

# ============================================================================
# PRINT STATISTICAL SUMMARY TABLE (for easy copy-paste)
# ============================================================================

# Calculate standard deviations from actual error data
sd_train = [train_errors[:, i].std() for i in range(6)]
sd_train.append(train_errors.flatten().std())  # Overall SD

sd_val = [val_errors[:, i].std() for i in range(6)]
sd_val.append(val_errors.flatten().std())  # Overall SD

# Statistical data from your paper + calculated SD
layer_names_clean = ['Layer 1 (5cm)', 'Layer 2 (15cm)', 'Layer 3 (25cm)', 
                     'Layer 4 (35cm)', 'Layer 5 (45cm)', 'Layer 6 (55cm)', 'Overall']
rmse_train = [2.06, 0.92, 0.74, 0.44, 0.75, 1.45, 1.19]
nrmse_train = [0.100, 0.089, 0.131, 0.119, 0.087, 0.087, 0.058]
r2_train = [0.657, 0.815, 0.523, 0.737, 0.874, 0.778, 0.781]
rmse_val = [2.29, 1.05, 0.98, 0.25, 1.18, 0.82, 1.25]
nrmse_val = [0.13, 0.17, 0.31, 0.17, 0.90, 1.52, 0.06]
r2_val = [0.660, 0.467, -0.311, 0.608, -11.630, -22.706, 0.798]

# Print formatted table
print("\n" + "="*130)
print("STATISTICAL SUMMARY - ERROR ANALYSIS")
print("="*130)
print(f"\n{'Soil Layer':<16} | {'TRAINING':^50} | {'FORECAST (VALIDATION)':^50}")
print(f"{'':16} | {'RMSE':>8} {'SD':>8} {'NRMSE':>10} {'R²':>10} | {'RMSE':>8} {'SD':>8} {'NRMSE':>10} {'R²':>10}")
print(f"{'':16} | {'(%VWC)':>8} {'(%VWC)':>8} {'':>10} {'':>10} | {'(%VWC)':>8} {'(%VWC)':>8} {'':>10} {'':>10}")
print("-"*130)

for i in range(7):
    layer = layer_names_clean[i]
    print(f"{layer:<16} | {rmse_train[i]:>8.2f} {sd_train[i]:>8.2f} {nrmse_train[i]:>10.3f} {r2_train[i]:>10.3f} | "
          f"{rmse_val[i]:>8.2f} {sd_val[i]:>8.2f} {nrmse_val[i]:>10.2f} {r2_val[i]:>10.3f}")

print("="*130)
print("\nNote: RMSE and SD in %VWC (volumetric water content percentage)")
print("      Dashed lines in box plots show ±1 standard deviation from the mean")
print("="*130 + "\n")

# Save and display the error analysis figure
# ENHANCED: Updated height to 1500 (50% increase from 1000)
output_path_error = "error_analysis_heatmap_boxplot.pdf"
fig_error_heatmap.write_image(output_path_error, format="pdf", width=4800, height=1500, scale=2)

print(f"✓ Error analysis figure saved to {output_path_error}\n")

fig_error_heatmap.show()


Error range for heatmap (measured - predicted):
  Min error: -2.511 %VWC (negative = predicted too high)
  Max error: 11.093 %VWC (positive = predicted too low)
  Zero position in colorscale: 0.185

STATISTICAL SUMMARY - ERROR ANALYSIS

Soil Layer       |                      TRAINING                      |               FORECAST (VALIDATION)               
                 |     RMSE       SD      NRMSE         R² |     RMSE       SD      NRMSE         R²
                 |   (%VWC)   (%VWC)                       |   (%VWC)   (%VWC)                      
----------------------------------------------------------------------------------------------------------------------------------
Layer 1 (5cm)    |     2.06     1.81      0.100      0.657 |     2.29     1.80       0.13      0.660
Layer 2 (15cm)   |     0.92     0.90      0.089      0.815 |     1.05     0.99       0.17      0.467
Layer 3 (25cm)   |     0.74     0.73      0.131      0.523 |     0.98     0.84       0.31     -0.311
Laye

## 8. Figure 3 — Estimated PINN Flow Functions
Generates `pinn_functions_vertical_by_layer.pdf`

Displays the estimated physical functions for each soil layer (rows) organized by function type (columns): rain infiltration `fr_i(r)`, vertical flow `f_i(m_i)`, inter-layer flow `f_{i,i+1}`, plant uptake `S_i(m_i, ET₀)`, lateral outflow `fl_i(m_i)`, and irrigation `f_irr(q_irr)` (layer 1 only). Blue-shaded regions indicate the measured soil moisture training range.

In [14]:
# Input-output relationships (unchanged from original)
m_range = np.linspace(0, 1, 100).reshape(-1, 1)
q_range = np.linspace(0, 1, 100).reshape(-1, 1)
r_range = np.linspace(0, 1, 100).reshape(-1, 1)
et0_range = np.linspace(0, 1, 100).reshape(-1, 1)
m_grid, et0_grid = np.meshgrid(m_range, et0_range)
m1_grid, m2_grid = np.meshgrid(m_range, m_range)

# Compute outputs (unchanged)
f_outputs = [getattr(pinn_model, f'f{i+1}_out')(getattr(pinn_model, f'f{i+1}_dense3')(getattr(pinn_model, f'f{i+1}_dense2')(getattr(pinn_model, f'f{i+1}_dense1')(m_range)))).numpy().flatten() for i in range(6)]
inflow_pinn = pinn_model.inflow_out(pinn_model.inflow_dense3(pinn_model.inflow_dense2(pinn_model.inflow_dense1(q_range)))).numpy().flatten()
in_rain_outputs = [getattr(pinn_model, f'rain{i+1}_out')(getattr(pinn_model, f'rain{i+1}_dense3')(getattr(pinn_model, f'rain{i+1}_dense2')(getattr(pinn_model, f'rain{i+1}_dense1')(r_range)))).numpy().flatten() for i in range(6)]

f_inter_grids = []
for flow in ['f12', 'f23', 'f34', 'f45', 'f56']:
    m12_input = np.stack([m1_grid.ravel(), m2_grid.ravel()], axis=1)
    layer = getattr(pinn_model, f'{flow}_dense1')(m12_input)
    for j in range(2, 4):
        layer = getattr(pinn_model, f'{flow}_dense{j}')(layer)
    f_out = getattr(pinn_model, f'{flow}_out')(layer).numpy().reshape(m1_grid.shape)
    f_inter_grids.append(f_out)

pc_grids = []
for i in range(1, 7):
    pc_input = np.stack([m_grid.ravel(), et0_grid.ravel()], axis=1)
    layer = getattr(pinn_model, f'pc{i}_dense1')(pc_input)
    for j in range(2, 5):
        layer = getattr(pinn_model, f'pc{i}_dense{j}')(layer)
    pc_out = getattr(pinn_model, f'pc{i}_out')(layer).numpy().reshape(m_grid.shape)
    pc_grids.append(pc_out)

m_range_tf = tf.constant(m_range, dtype=tf.float64)
fl_outputs = []
for i in range(1, 6):
    x = getattr(pinn_model, f'fl{i}_dense1')(m_range_tf)
    x = getattr(pinn_model, f'fl{i}_dense2')(x)
    x = getattr(pinn_model, f'fl{i}_dense3')(x)
    fl = getattr(pinn_model, f'fl{i}_out')(x).numpy().flatten()
    fl_outputs.append(fl)

# ── Layer moisture bounds for training-range shading ──
# (used in Figure 3 to highlight the measured data range)
layer_bounds = {
    k: (float(df_train[f'moisture_{k}'].min()),
        float(df_train[f'moisture_{k}'].max()))
    for k in range(1, 7)
}

# Redefine as 1-D arrays for Figure 3 line/contour plots
m_range   = np.linspace(0, 1, 100)
q_range   = np.linspace(0, 1, 100)
r_range   = np.linspace(0, 1, 100)
et0_range = np.linspace(0, 1, 100)

print('✓ PINN flow functions computed and ranges defined')


✓ PINN flow functions computed and ranges defined


In [15]:
# ALTERNATIVE: Vertical organization by soil layer (each row = one layer)
# Columns: [Rain, Vertical out, Inter-layer, Plant uptake, Lateral, Irrigation(row1 only)]

# ═══════════════════════════════════════════════════════════
# FONT SCALING CONTROL - Adjust this value to change all font sizes
# ═══════════════════════════════════════════════════════════
# Change this single value to scale ALL fonts in the figure:
# - Axis tick labels
# - Axis titles  
# - Colorbar titles
# Try values: 1.5 (smaller), 2.0 (default), 2.5 (larger), 3.0 (very large)
FONT_SCALE = 3.0

# 6 rows (one per layer), 6 columns
specs = []
# Row 1 (Layer 1): Has irrigation in last column
specs.append([{"type": "xy"}, {"type": "xy"}, {"type": "contour"}, 
              {"type": "contour"}, {"type": "xy"}, {"type": "xy"}])
# Rows 2-5 (Layers 2-5): All functions
for i in range(4):
    specs.append([{"type": "xy"}, {"type": "xy"}, {"type": "contour"}, 
                  {"type": "contour"}, {"type": "xy"}, None])
# Row 6 (Layer 6): No inter-layer flow below, no lateral for layer 6
specs.append([{"type": "xy"}, {"type": "xy"}, None, 
              {"type": "contour"}, None, None])

fig_vert = make_subplots(
    rows=6, cols=6,
    row_heights=[0.17, 0.17, 0.17, 0.17, 0.17, 0.15],
    column_widths=[0.15, 0.15, 0.20, 0.20, 0.15, 0.15],
    subplot_titles=[""] * 36,  # Clean, no titles
    specs=specs,
    horizontal_spacing=0.06,  # Increased from 0.035 to give more space between columns
    vertical_spacing=0.040
)

# ═══════════════════════════════════════════════════════════
# POPULATE EACH LAYER ROW
# ═══════════════════════════════════════════════════════════
for layer in range(1, 7):
    row = layer
    
    # Col 1: Rain inflow
    fig_vert.add_trace(
        go.Scatter(x=30 * r_range.squeeze(), y=40 * in_rain_outputs[layer-1],
                   mode="lines", line=dict(color="blue", width=4), showlegend=False),
        row=row, col=1
    )
    
    # Col 2: Vertical outflow f_i
    fig_vert.add_trace(
        go.Scatter(x=40 * m_range.squeeze(), y=40 * f_outputs[layer-1],
                   mode="lines", line=dict(color="blue", width=4), showlegend=False),
        row=row, col=2
    )
    if layer in layer_bounds:
        vmin, vmax = layer_bounds[layer]
        fig_vert.add_vrect(x0=vmin, x1=vmax, row=row, col=2,
                          fillcolor="rgba(0,150,200,0.15)", line_width=0, layer="below")
    
    # Col 3: Inter-layer flow f_i,i+1 (only for layers 1-5)
    if layer < 6:
        # Calculate global min/max for all inter-layer flows
        z_values_inter = [40 * grid for grid in f_inter_grids]
        zmin_inter = min(np.min(z) for z in z_values_inter)
        zmax_inter = max(np.max(z) for z in z_values_inter)
        ncontours = 30
        
        fig_vert.add_trace(
            go.Contour(x=40 * np.array(m_range), y=40 * np.array(m_range),
                       z=40 * f_inter_grids[layer-1], coloraxis="coloraxis1",
                       contours=dict(
                           start=zmin_inter,
                           end=zmax_inter,
                           size=(zmax_inter - zmin_inter) / ncontours,
                           showlines=False
                       ), showscale=(layer == 5)),
            row=row, col=3
        )
        # Add training domain rectangle
        try:
            if isinstance(layer_bounds, dict):
                x0, x1 = layer_bounds[layer]
                y0, y1 = layer_bounds[layer + 1]
                if all(v is not None for v in [x0, x1, y0, y1]):
                    fig_vert.add_shape(
                        type="rect", x0=x0, x1=x1, y0=y0, y1=y1,
                        fillcolor="rgba(0,150,200,0.5)",
                        line_width=0, layer="above", row=row, col=3
                    )
        except (KeyError, TypeError, IndexError):
            pass
    
    # Col 4: Plant uptake S_i
    # Calculate global min/max for all plant uptake
    z_values_pc = [40 * grid for grid in pc_grids]
    zmin_pc = min(np.min(z) for z in z_values_pc)
    zmax_pc = max(np.max(z) for z in z_values_pc)
    ncontours = 30
    
    fig_vert.add_trace(
        go.Contour(x=40 * np.array(m_range), y=0.8 * np.array(et0_range),
                   z=40 * pc_grids[layer-1], coloraxis="coloraxis2",
                   contours=dict(
                       start=zmin_pc,
                       end=zmax_pc,
                       size=(zmax_pc - zmin_pc) / ncontours,
                       showlines=False
                   ), showscale=(layer == 6)),
        row=row, col=4
    )
    # Add training domain rectangle
    try:
        if isinstance(layer_bounds, dict):
            x0, x1 = layer_bounds[layer]
            if x0 is not None and x1 is not None:
                fig_vert.add_shape(
                    type="rect", x0=x0, x1=x1, y0=0.0, y1=0.8,
                    fillcolor="rgba(0,150,200,0.5)",
                    line_width=0, layer="above", row=row, col=4
                )
    except (KeyError, TypeError, IndexError):
        pass
    
    # Col 5: Lateral outflow fl_i (only layers 1-5)
    if layer <= 5:
        fig_vert.add_trace(
            go.Scatter(x=40 * m_range.squeeze(), y=40 * fl_outputs[layer-1],
                       mode="lines", line=dict(color="blue", width=4), showlegend=False),
            row=row, col=5
        )
        if layer in layer_bounds:
            x0, x1 = layer_bounds[layer]
            fig_vert.add_vrect(x0=x0, x1=x1, row=row, col=5,
                              fillcolor="rgba(0,150,200,0.15)", line_width=0, layer="below")

# Col 6, Row 1: IRRIGATION (only in layer 1, rightmost)
fig_vert.add_trace(
    go.Scatter(x=np.squeeze(q_range), y=40 * inflow_pinn,
               mode="lines", line=dict(color="blue", width=4), showlegend=False),
    row=1, col=6
)

# ═══════════════════════════════════════════════════════════
# COLORBARS - HORIZONTAL ABOVE/BELOW COLUMNS
# ═══════════════════════════════════════════════════════════
z_values_inter = [40 * grid for grid in f_inter_grids]
zmin_inter = min(np.min(z) for z in z_values_inter)
zmax_inter = max(np.max(z) for z in z_values_inter)
z_values_pc = [40 * grid for grid in pc_grids]
zmin_pc = min(np.min(z) for z in z_values_pc)
zmax_pc = max(np.max(z) for z in z_values_pc)

fig_vert.update_layout(
    coloraxis1=dict(
        colorscale="Viridis", cmin=zmin_inter, cmax=zmax_inter,
        colorbar=dict(
            title="Inter-layer flow (%VWC/h)",
            orientation="h",  # Horizontal
            x=0.4, xanchor="center",  # Center on column 3
            y=0.1, yanchor="top",  # Below the plots
            len=0.16,  # Length matches column width
            thickness=12,
            tickfont=dict(size=int(10*FONT_SCALE)),
            title_font=dict(size=int(11*FONT_SCALE)),
            title_side="bottom"  # Title below the colorbar
        )
    ),
    coloraxis2=dict(
        colorscale="Viridis", cmin=zmin_pc, cmax=zmax_pc,
        colorbar=dict(
            title="Plant uptake (%VWC/h)",
            orientation="h",  # Horizontal
            x=0.6, xanchor="center",  # Center on column 4 (updated position)
            y=-0.04, yanchor="top",  # Below the plots
            len=0.16,  # Length matches column width
            thickness=12,
            tickfont=dict(size=int(10*FONT_SCALE)),
            title_font=dict(size=int(11*FONT_SCALE)),
            title_side="bottom"  # Title below the colorbar
        )
    )
)

# ═══════════════════════════════════════════════════════════
# AXIS CONFIGURATION - INDEPENDENT AXES WITH LABELS ON EACH SUBPLOT
# ═══════════════════════════════════════════════════════════
# Each subplot gets its own independent axes with full labels

# Column 1: Rain inflow
for row in range(1, 7):
    fig_vert.update_xaxes(title_text="Rain (mm/h)", showticklabels=True,
                         tickfont=dict(size=int(10*FONT_SCALE)),
                         title_font=dict(size=int(11*FONT_SCALE)),
                         title_standoff=5,
                         row=row, col=1)
    fig_vert.update_yaxes(title_text="fr<sub>%d</sub> (%% VWC/h)" % row, showticklabels=True,
                         tickfont=dict(size=int(10*FONT_SCALE)),
                         title_font=dict(size=int(11*FONT_SCALE)),
                         title_standoff=5,
                         row=row, col=1)

# Column 2: Vertical outflow
for row in range(1, 7):
    fig_vert.update_xaxes(title_text="m<sub>%d</sub> (%% VWC)" % row, showticklabels=True,
                         tickfont=dict(size=int(10*FONT_SCALE)),
                         title_font=dict(size=int(11*FONT_SCALE)),
                         title_standoff=5,
                         row=row, col=2)
    fig_vert.update_yaxes(title_text="f<sub>%d</sub> (%%VWC/h)" % row, showticklabels=True,
                         tickfont=dict(size=int(10*FONT_SCALE)),
                         title_font=dict(size=int(11*FONT_SCALE)),
                         title_standoff=5,
                         row=row, col=2)

# Column 3: Inter-layer flow (only rows 1-5)
for row in range(1, 6):
    fig_vert.update_xaxes(title_text="m<sub>%d</sub> (%% VWC)" % row, showticklabels=True,
                         tickfont=dict(size=int(10*FONT_SCALE)),
                         title_font=dict(size=int(11*FONT_SCALE)),
                         title_standoff=5,
                         row=row, col=3)
    fig_vert.update_yaxes(title_text="m<sub>%d</sub> (%% VWC)" % (row+1), showticklabels=True,
                         tickfont=dict(size=int(10*FONT_SCALE)),
                         title_font=dict(size=int(11*FONT_SCALE)),
                         title_standoff=5,
                         row=row, col=3)

# Column 4: Plant uptake
for row in range(1, 7):
    fig_vert.update_xaxes(title_text="m<sub>%d</sub> (%% VWC)" % row, showticklabels=True,
                         tickfont=dict(size=int(10*FONT_SCALE)),
                         title_font=dict(size=int(11*FONT_SCALE)),
                         title_standoff=5,
                         row=row, col=4)
    fig_vert.update_yaxes(title_text="ET<sub>0</sub> (mm/h)", showticklabels=True,
                         tickfont=dict(size=int(10*FONT_SCALE)),
                         title_font=dict(size=int(11*FONT_SCALE)),
                         title_standoff=5,
                         row=row, col=4)

# Column 5: Lateral outflow (only rows 1-5)
for row in range(1, 6):
    fig_vert.update_xaxes(title_text="m<sub>%d</sub> (%% VWC)" % row, showticklabels=True,
                         tickfont=dict(size=int(10*FONT_SCALE)),
                         title_font=dict(size=int(11*FONT_SCALE)),
                         title_standoff=5,
                         row=row, col=5)
    fig_vert.update_yaxes(title_text="fl<sub>%d</sub> (%% VWC/h)" % row, showticklabels=True,
                         tickfont=dict(size=int(10*FONT_SCALE)),
                         title_font=dict(size=int(11*FONT_SCALE)),
                         title_standoff=5,
                         row=row, col=5)

# Column 6: Irrigation (only row 1)
fig_vert.update_xaxes(title_text="Irrigation Time (h/h)", showticklabels=True,
                     tickfont=dict(size=int(10*FONT_SCALE)),
                     title_font=dict(size=int(11*FONT_SCALE)),
                     title_standoff=5,
                     row=1, col=6)
fig_vert.update_yaxes(title_text="Inflow (% VWC/h)", showticklabels=True,
                     tickfont=dict(size=int(10*FONT_SCALE)),
                     title_font=dict(size=int(11*FONT_SCALE)),
                     title_standoff=1,
                     row=1, col=6)

# Add column headers - centered on each column
# Column widths: [0.15, 0.15, 0.20, 0.20, 0.15, 0.15], horizontal_spacing: 0.06
col_headers = ["Rain", "Vertical\nflow", "Inter-layer\nflow", 
               "Plant\nuptake", "Lateral\noutflow", "Irrigation"]
# Calculated center positions for each column
col_positions = [0.07, 0.23, 0.4, 0.6, 0.78, 0.955]

for label, x_pos in zip(col_headers, col_positions):
    fig_vert.add_annotation(
        text=f"<b>{label}</b>", xref="paper", yref="paper",
        x=x_pos, y=1.01, xanchor="center", yanchor="bottom",
        showarrow=False, font=dict(size=int(14*FONT_SCALE), family="Arial")
    )

# Add row labels (Layers 2-6) on the right side - skip Layer 1 (has irrigation)
# Use <br> for line breaks in the labels
layer_labels = ["Estimated Flows<br>at 15cm", "Estimated Flows<br>at 25cm", 
                "Estimated Flows<br>at 35cm", "Estimated Flows<br>at 45cm", "Estimated Flows<br>at 55cm"]
layer_y_positions = [0.73, 0.56, 0.39, 0.22, 0.07]  # Skip first position (Layer 1)

for label, y_pos in zip(layer_labels, layer_y_positions):
    fig_vert.add_annotation(
        text=f"<b>{label}</b>", xref="paper", yref="paper",
        x=0.9, y=y_pos, xanchor="left", yanchor="middle",
        showarrow=False, font=dict(size=int(14*FONT_SCALE), family="Arial")
    )

# ═══════════════════════════════════════════════════════════
# LAYOUT
# ═══════════════════════════════════════════════════════════
# Apply automargin to all y-axes to bring titles closer
fig_vert.update_yaxes(automargin=True)

fig_vert.update_layout(
    template="plotly_white",
    width=3600, height=2400,
    margin=dict(l=80, r=200, t=60, b=80),  # Reduced left margin from 150 to 80
    font=dict(family="Arial", size=int(14*FONT_SCALE)),
    showlegend=False
)

output_path_vertical = "pinn_functions_vertical_by_layer.pdf"
fig_vert.write_image(output_path_vertical, format="pdf", width=3600, height=2400, scale=2)
print(f"✓ Vertical by layer figure: {output_path_vertical}")
print(f"  ✓ Independent axes - each subplot has its own scale")
print(f"  ✓ Full axis labels on every subplot")
print(f"  ✓ X and Y axis titles showing function and variable names")
print(f"  ✓ All plots analyzable individually")
fig_vert.show()

✓ Vertical by layer figure: pinn_functions_vertical_by_layer.pdf
  ✓ Independent axes - each subplot has its own scale
  ✓ Full axis labels on every subplot
  ✓ X and Y axis titles showing function and variable names
  ✓ All plots analyzable individually
